# From-scratch Turkish BPE + dialogue pilot

This notebook trains a byte-level BPE tokenizer and a small causal language model **from random initialization** in two stages: Common Voice Turkish train text (CC0-1.0) for language pretraining, then the 2,480 train-only Turkish Daily Dialogues conversations (CC BY 4.0, still marked `draft`) for dialogue fine-tuning. No pretrained model weights are used.

## Run it
1. Import this notebook from `dozagoat47-star/NextgenAI-Clean-Kaggle-Pilot` in Kaggle.
2. Enable **Internet** and select a GPU accelerator.
3. Choose **Run All**. Setup checks packaged file hashes and confirms that only Common Voice train/validation files are included.

The notebook never evaluates either held-out test split. Validation is used for early stopping/checkpoint selection. Common Voice is Wikipedia-weighted, and the dialogue set is small, synthetic, and not individually human-reviewed. Treat the output as an experiment, not a usable chat model.

In [ ]:
from datetime import datetime, timezone
import hashlib
import json
from pathlib import Path
import subprocess
import sys
from zipfile import ZipFile

WORK_ROOT = Path('/kaggle/working') / (
    'nextgenai_two_stage_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
)
REPOSITORY_URL = 'https://github.com/dozagoat47-star/NextgenAI-Clean-Kaggle-Pilot.git'
if WORK_ROOT.exists():
    raise FileExistsError(f'Refusing to reuse existing work directory: {WORK_ROOT}')
subprocess.run(
    ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(WORK_ROOT)],
    check=True,
)
bundle = json.loads((WORK_ROOT / 'kaggle_manifest.json').read_text(encoding='utf-8'))
if bundle.get('review_status') != 'draft':
    raise ValueError('Expected draft-status dialogue data')
for relative_name, expected_hash in bundle.get('files', {}).items():
    relative_path = Path(relative_name)
    if relative_path.is_absolute() or '..' in relative_path.parts:
        raise ValueError(f'Unsafe manifest path: {relative_name}')
    content = (WORK_ROOT / relative_path).read_bytes()
    if relative_path.suffix.lower() in {'.md', '.py', '.txt', '.json', '.ipynb'}:
        content = content.replace(b'\r\n', b'\n')
    actual_hash = hashlib.sha256(content).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f'GitHub file checksum mismatch: {relative_name}')

dialogue_path = WORK_ROOT / 'data/raw/turkish_daily_dialogues_sft_001/train.jsonl'
dialogue_manifest_path = WORK_ROOT / 'data/raw/turkish_daily_dialogues_sft_001/manifest.json'
dialogue_manifest = json.loads(dialogue_manifest_path.read_text(encoding='utf-8'))
if hashlib.sha256(dialogue_path.read_bytes()).hexdigest() != dialogue_manifest.get('training_data_sha256'):
    raise ValueError('Dialogue data does not match its manifest')
if dialogue_manifest.get('source_selection', {}).get('source_split') != 'train only':
    raise ValueError('Dialogue input is not upstream train-only data')

cv = bundle.get('common_voice', {})
if cv.get('license') != 'CC0-1.0' or cv.get('test_included') is not False:
    raise ValueError('Expected CC0 Common Voice train/validation only')
archive_relative = Path(cv.get('archive_path', ''))
if archive_relative.is_absolute() or '..' in archive_relative.parts:
    raise ValueError('Unsafe Common Voice archive path')
archive_path = WORK_ROOT / archive_relative
corpus_dir = WORK_ROOT / 'data/raw/common_voice_27_active_trainval_001'
corpus_dir.mkdir(parents=True, exist_ok=False)
with ZipFile(archive_path) as archive:
    names = set(archive.namelist())
    if names != {'train.tsv', 'validation.tsv', 'manifest.json'}:
        raise ValueError(f'Unexpected Common Voice archive contents: {sorted(names)}')
    for name in sorted(names):
        (corpus_dir / name).write_bytes(archive.read(name))
expected_corpus_files = {'train.tsv', 'validation.tsv', 'manifest.json'}
if set(cv.get('extracted_sha256', {})) != expected_corpus_files:
    raise ValueError('Unexpected Common Voice checksum manifest')
for name, expected_hash in cv['extracted_sha256'].items():
    actual_hash = hashlib.sha256((corpus_dir / name).read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f'Common Voice file checksum mismatch: {name}')
if (corpus_dir / 'test.tsv').exists():
    raise ValueError('The Common Voice test split must not be present')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'tokenizers>=0.20,<1.0'],
    check=True,
)
sys.path.insert(0, str(WORK_ROOT))
import torch
from clean_model.data import load_jsonl, split_by_group
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU not detected. Enable a Kaggle GPU accelerator and restart the session.')
records = load_jsonl(dialogue_path, require_approved=False)
if not records or any(row['review']['status'] != 'draft' for row in records):
    raise ValueError('Expected explicitly draft-only dialogue data')
splits = split_by_group(records, seed=7, require_approved=False)
print('GitHub source:', REPOSITORY_URL)
print('Common Voice:', cv['dataset'], '| CC0-1.0 | train/validation only')
print('Dialogues:', len(records), '| CC BY 4.0 | draft | train split only')
print('Dialogue internal splits:', {name: len(rows) for name, rows in splits.items()})
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
print('Both held-out test splits remain untouched.')

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import sys

SEED = 7
VOCAB_SIZE = 2048
PRETRAIN_EPOCHS = 4
FINETUNE_EPOCHS = 12
PATIENCE = 3
BATCH_SIZE = 16
D_MODEL = 128
NUM_HEADS = 4
NUM_LAYERS = 4
MAX_SEQ_LEN = 256
TRAIN_TOKEN_BUDGET = 2000000
VALIDATION_TOKEN_BUDGET = 250000

OUTPUT_DIR = Path('/kaggle/working') / (
    'turkish_bpe_chat_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
)
command = [
    sys.executable, '-m', 'clean_model.two_stage',
    '--common-voice', str(corpus_dir),
    '--dialogue-data', str(dialogue_path),
    '--output', str(OUTPUT_DIR),
    '--seed', str(SEED),
    '--vocab-size', str(VOCAB_SIZE),
    '--pretrain-epochs', str(PRETRAIN_EPOCHS),
    '--finetune-epochs', str(FINETUNE_EPOCHS),
    '--patience', str(PATIENCE),
    '--batch-size', str(BATCH_SIZE),
    '--d-model', str(D_MODEL),
    '--num-heads', str(NUM_HEADS),
    '--num-layers', str(NUM_LAYERS),
    '--max-seq-len', str(MAX_SEQ_LEN),
    '--train-token-budget', str(TRAIN_TOKEN_BUDGET),
    '--validation-token-budget', str(VALIDATION_TOKEN_BUDGET),
    '--allow-draft-data',
]
print('Output directory:', OUTPUT_DIR)
print('This starts a new from-scratch run; it does not resume the old checkpoint.')

In [ ]:
subprocess.run(command, cwd=WORK_ROOT, check=True)

In [ ]:
metadata = json.loads((OUTPUT_DIR / 'training_metadata.json').read_text(encoding='utf-8'))
summary = {
    'checkpoint': str(OUTPUT_DIR / 'pilot_model.pt'),
    'tokenizer': metadata['tokenizer'],
    'common_voice_train_sentences': metadata['data']['common_voice_train_sentences_selected'],
    'dialogue_split_counts': metadata['data']['dialogue_split_counts'],
    'pretraining_best_epoch': metadata['pretraining']['best_epoch'],
    'pretraining_validation_loss': metadata['pretraining']['best_validation_loss'],
    'finetune_best_epoch': metadata['dialogue_finetuning']['best_epoch'],
    'finetune_validation_loss': metadata['dialogue_finetuning']['best_validation_loss'],
    'heldout_test_evaluated': metadata['test_split_evaluated'],
    'checkpoint_sha256': metadata['checkpoint_sha256'],
    'interpretation': metadata['interpretation'],
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
samples = json.loads((OUTPUT_DIR / 'generation_samples.json').read_text(encoding='utf-8'))
for sample in samples:
    print('\nPROMPT:', sample['context'][-1]['content'])
    print('REFERENCE:', sample['reference'])
    print('GENERATED:', sample['generated'])

The output folder is under `/kaggle/working`. Download `pilot_model.pt` and `training_metadata.json`; `generation_samples.json` is useful for reviewing answers. The tokenizer is embedded in the checkpoint and also saved as `tokenizer.json`. This pilot is not production-ready, and validation loss is not a measure of conversational usefulness.